# PyTorch를 활용한 MNIST 데이터셋 분석

본 노트북에서는 MNIST 손글씨 숫자 데이터셋을 PyTorch로 불러와
다음 단계를 순차적으로 수행합니다.

1. 라이브러리 임포트 및 환경 설정
2. 데이터 로드
3. EDA (탐색적 데이터 분석)
4. 시각화
5. 전처리 (DataLoader 구성)
6. 모델 정의 및 학습
7. 모델 평가

---
## 1. 라이브러리 임포트 및 환경 설정

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix

# 한글 폰트 설정 (Windows)
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# 랜덤 시드 고정 (재현성 확보)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# 디바이스 설정 (GPU 우선, 없으면 CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')

---
## 2. 데이터 로드

- `torchvision.datasets.MNIST`를 통해 훈련/테스트 데이터셋을 자동으로 다운로드
- 기본 변환: PIL 이미지 → Tensor (값 범위 0~1)

In [ ]:
# 기본 변환 (텐서 변환만 적용 — EDA 목적)
basic_transform = transforms.ToTensor()

# 훈련 데이터 (60,000장)
train_dataset = datasets.MNIST(
    root='./datasets',
    train=True,
    download=True,
    transform=basic_transform
)

# 테스트 데이터 (10,000장)
test_dataset = datasets.MNIST(
    root='./datasets',
    train=False,
    download=True,
    transform=basic_transform
)

print(f'훈련 데이터 크기: {len(train_dataset)}')
print(f'테스트 데이터 크기: {len(test_dataset)}')
print(f'이미지 shape (단일 샘플): {train_dataset[0][0].shape}')  # (C, H, W)
print(f'클래스 종류: {train_dataset.classes}')

---
## 3. EDA (탐색적 데이터 분석)

데이터의 기본 통계, 클래스 분포 등을 확인합니다.

In [ ]:
# ── 클래스 분포 확인 ──────────────────────────────
train_labels = train_dataset.targets.numpy()
test_labels  = test_dataset.targets.numpy()

train_counter = Counter(train_labels)
test_counter  = Counter(test_labels)

print('=== 훈련 데이터 클래스별 샘플 수 ===')
for cls in sorted(train_counter):
    print(f'  숫자 {cls}: {train_counter[cls]}장')

print('\n=== 테스트 데이터 클래스별 샘플 수 ===')
for cls in sorted(test_counter):
    print(f'  숫자 {cls}: {test_counter[cls]}장')

In [ ]:
# ── 픽셀 값 기초 통계 ─────────────────────────────
# 전체 훈련 이미지를 numpy 배열로 변환
train_images = train_dataset.data.numpy()  # shape: (60000, 28, 28), 값 범위 0~255

print('=== 픽셀 값 기초 통계 (0~255 범위) ===')
print(f'  최솟값: {train_images.min()}')
print(f'  최댓값: {train_images.max()}')
print(f'  평균  : {train_images.mean():.4f}')
print(f'  표준편차: {train_images.std():.4f}')

---
## 4. 시각화

In [ ]:
# ── 클래스별 샘플 이미지 시각화 ───────────────────
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('클래스별 대표 샘플 이미지 (0~9)', fontsize=14, fontweight='bold')

for digit in range(10):
    # 해당 숫자 클래스의 첫 번째 인덱스 찾기
    idx = (train_dataset.targets == digit).nonzero(as_tuple=True)[0][0].item()
    img, label = train_dataset[idx]
    ax = axes[digit // 5][digit % 5]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'숫자: {label}', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ── 클래스 분포 막대 그래프 ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, counter, title in zip(
    axes,
    [train_counter, test_counter],
    ['훈련 데이터 클래스 분포', '테스트 데이터 클래스 분포']
):
    classes = sorted(counter.keys())
    counts  = [counter[c] for c in classes]
    bars = ax.bar(classes, counts, color=plt.cm.tab10.colors, edgecolor='black')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('숫자 레이블')
    ax.set_ylabel('샘플 수')
    ax.set_xticks(classes)
    # 막대 위에 수치 표시
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                str(cnt), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── 픽셀 밝기 분포 히스토그램 ─────────────────────
plt.figure(figsize=(9, 4))
plt.hist(train_images.flatten(), bins=50, color='steelblue', edgecolor='white', alpha=0.8)
plt.title('훈련 데이터 전체 픽셀 밝기 분포', fontsize=13, fontweight='bold')
plt.xlabel('픽셀 값 (0~255)')
plt.ylabel('빈도')
plt.tight_layout()
plt.show()

In [ ]:
# ── 클래스별 평균 이미지 시각화 ───────────────────
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('클래스별 평균 이미지', fontsize=14, fontweight='bold')

for digit in range(10):
    mask = train_dataset.targets == digit
    mean_img = train_dataset.data[mask].float().mean(dim=0).numpy()
    ax = axes[digit // 5][digit % 5]
    im = ax.imshow(mean_img, cmap='hot')
    ax.set_title(f'숫자: {digit}', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 5. 전처리 및 DataLoader 구성

- 픽셀 값을 정규화: `mean=0.1307`, `std=0.3081` (MNIST 공식 통계)
- 훈련 데이터에서 검증셋 10% 분리
- `DataLoader`를 생성하여 배치 단위 학습 준비

In [ ]:
# MNIST 공식 평균/표준편차로 정규화
MEAN = 0.1307
STD  = 0.3081

final_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,))
])

# 정규화 변환을 적용한 새 데이터셋
train_dataset_norm = datasets.MNIST(root='./datasets', train=True,  download=False, transform=final_transform)
test_dataset_norm  = datasets.MNIST(root='./datasets', train=False, download=False, transform=final_transform)

# 훈련셋 → 훈련(90%) / 검증(10%) 분리
val_size   = int(len(train_dataset_norm) * 0.1)
train_size = len(train_dataset_norm) - val_size
train_set, val_set = random_split(train_dataset_norm, [train_size, val_size],
                                  generator=torch.Generator().manual_seed(SEED))

BATCH_SIZE = 64

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset_norm, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'훈련 샘플: {train_size} | 검증 샘플: {val_size} | 테스트 샘플: {len(test_dataset_norm)}')
print(f'배치 크기: {BATCH_SIZE}')
print(f'훈련 배치 수: {len(train_loader)} | 검증 배치 수: {len(val_loader)}')

In [ ]:
# ── 정규화 후 샘플 확인 ─────────────────────────
sample_imgs, sample_labels = next(iter(train_loader))
print(f'배치 이미지 shape: {sample_imgs.shape}')   # (64, 1, 28, 28)
print(f'정규화 후 픽셀 범위: [{sample_imgs.min():.3f}, {sample_imgs.max():.3f}]')
print(f'배치 레이블 shape: {sample_labels.shape}')

---
## 6. 모델 정의 및 학습

### 모델 구조
- **CNN (Convolutional Neural Network)** 사용
  - Conv2d → BatchNorm → ReLU → MaxPool 블록 × 2
  - Dropout으로 과적합 방지
  - Fully Connected 레이어로 최종 분류

### 학습 설정
- Optimizer: Adam (lr=0.001)
- Loss: CrossEntropyLoss
- Epochs: 10
- 검증 손실 기준 Early Stopping (patience=3) 및 모델 체크포인트 저장

In [ ]:
class MnistCNN(nn.Module):
    """MNIST 분류용 CNN 모델"""

    def __init__(self, num_classes: int = 10):
        super(MnistCNN, self).__init__()

        # ── 특성 추출부 (Feature Extractor) ──────────
        self.features = nn.Sequential(
            # 블록 1: 1 채널 입력 → 32 채널
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   # (B, 32, 28, 28)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),         # (B, 32, 14, 14)
            nn.Dropout2d(p=0.25),

            # 블록 2: 32 채널 → 64 채널
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # (B, 64, 14, 14)
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),         # (B, 64, 7, 7)
            nn.Dropout2d(p=0.25),
        )

        # ── 분류부 (Classifier) ────────────────────
        self.classifier = nn.Sequential(
            nn.Flatten(),                   # (B, 64*7*7) = (B, 3136)
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(256, num_classes)     # 출력: 10개 클래스
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# 모델 인스턴스 생성 및 디바이스 이동
model = MnistCNN(num_classes=10).to(device)
print(model)

# 파라미터 수 출력
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n전체 파라미터 수: {total_params:,}')
print(f'학습 가능한 파라미터 수: {trainable_params:,}')

In [ ]:
# ── 학습 하이퍼파라미터 설정 ─────────────────────
EPOCHS    = 10
LR        = 0.001
PATIENCE  = 3          # 조기 종료 인내 횟수

optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

# 학습률 스케줄러: 검증 Loss 개선 없을 때 학습률을 0.5배 감소
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                  factor=0.5, patience=2, verbose=True)

# 학습 기록 저장
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

best_val_loss  = float('inf')
patience_count = 0
best_model_path = './models/mnist_best.pth'

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """한 에포크 학습 함수"""
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()           # 그래디언트 초기화
        outputs = model(images)         # 순전파
        loss = criterion(outputs, labels)  # 손실 계산
        loss.backward()                 # 역전파
        optimizer.step()                # 파라미터 업데이트

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += images.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    """검증/테스트 에포크 평가 함수"""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += images.size(0)

    return total_loss / total, correct / total

In [ ]:
import os
os.makedirs('./models', exist_ok=True)

print('=' * 65)
print(f'{'에포크':^6} | {'훈련 손실':^10} | {'훈련 정확도':^10} | {'검증 손실':^10} | {'검증 정확도':^10}')
print('=' * 65)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)

    # 스케줄러 업데이트
    scheduler.step(val_loss)

    # 기록 저장
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'{epoch:^6} | {train_loss:^10.4f} | {train_acc*100:^9.2f}% | {val_loss:^10.4f} | {val_acc*100:^9.2f}%')

    # 조기 종료 및 최적 모델 저장
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_count = 0
        torch.save(model.state_dict(), best_model_path)
        print(f'  → 최적 모델 저장 (검증 손실: {best_val_loss:.4f})')
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'\n[조기 종료] {epoch}에포크에서 학습 중단 (patience={PATIENCE})')
            break

print('=' * 65)
print('학습 완료!')

In [ ]:
# ── 학습 곡선 시각화 ──────────────────────────────
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 손실 곡선
axes[0].plot(epochs_ran, history['train_loss'], marker='o', label='훈련 손실')
axes[0].plot(epochs_ran, history['val_loss'],   marker='s', label='검증 손실')
axes[0].set_title('에포크별 손실 (Loss)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('에포크')
axes[0].set_ylabel('손실')
axes[0].legend()
axes[0].grid(True)

# 정확도 곡선
axes[1].plot(epochs_ran, [a*100 for a in history['train_acc']], marker='o', label='훈련 정확도')
axes[1].plot(epochs_ran, [a*100 for a in history['val_acc']],   marker='s', label='검증 정확도')
axes[1].set_title('에포크별 정확도 (Accuracy)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('에포크')
axes[1].set_ylabel('정확도 (%)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

---
## 7. 모델 평가

학습 시 저장된 최적 모델을 불러와 테스트셋으로 최종 평가를 수행합니다.

In [ ]:
# ── 최적 모델 로드 ────────────────────────────────
best_model = MnistCNN(num_classes=10).to(device)
best_model.load_state_dict(torch.load(best_model_path, map_location=device))

test_loss, test_acc = evaluate(best_model, test_loader, criterion, device)
print(f'테스트 손실: {test_loss:.4f}')
print(f'테스트 정확도: {test_acc*100:.2f}%')

In [ ]:
# ── 전체 예측 수집 ────────────────────────────────
all_preds, all_labels = [], []

best_model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = best_model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

In [ ]:
# ── 분류 리포트 출력 ──────────────────────────────
print('=== 분류 리포트 ===')
print(classification_report(all_labels, all_preds,
                             target_names=[str(i) for i in range(10)]))

In [ ]:
# ── 혼동 행렬 시각화 ──────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('혼동 행렬 (Confusion Matrix)', fontsize=14, fontweight='bold')
plt.xlabel('예측 레이블')
plt.ylabel('실제 레이블')
plt.tight_layout()
plt.show()

In [ ]:
# ── 오분류 샘플 시각화 ────────────────────────────
wrong_mask   = all_preds != all_labels
wrong_indices = np.where(wrong_mask)[0]

# 테스트셋 이미지 numpy 배열
test_images_np = test_dataset_norm.data.numpy()  # 원본 (비정규화) 이미지 사용

n_show = min(10, len(wrong_indices))
fig, axes = plt.subplots(2, 5, figsize=(13, 6))
fig.suptitle(f'오분류 샘플 (총 {wrong_mask.sum()}개 중 {n_show}개 표시)', fontsize=13, fontweight='bold')

for i, idx in enumerate(wrong_indices[:n_show]):
    ax = axes[i // 5][i % 5]
    ax.imshow(test_images_np[idx], cmap='gray')
    ax.set_title(f'실제: {all_labels[idx]}  예측: {all_preds[idx]}',
                 fontsize=10, color='red')
    ax.axis('off')

plt.tight_layout()
plt.show()

print(f'오분류 비율: {wrong_mask.sum()} / {len(all_labels)} = {wrong_mask.mean()*100:.2f}%')

---
## 결과 요약

| 항목 | 내용 |
|------|------|
| 데이터셋 | MNIST (훈련 60,000 / 테스트 10,000) |
| 모델 | CNN (Conv × 4, BN, Dropout, FC × 2) |
| 옵티마이저 | Adam (lr=0.001) |
| 정규화 | Batch Normalization + Dropout |
| 조기 종료 | patience=3 (검증 손실 기준) |
| 최종 테스트 정확도 | 셀 실행 후 확인 |

> 모델 파일은 `./models/mnist_best.pth` 경로에 저장됩니다.